# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading, exploring, and analyzing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library as described in its Croissant schema.

### Dataset Source
The dataset source is accessed via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata
print(f"{metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by the Croissant schema.

In Croissant, a **RecordSet** is a table or logical collection of records (akin to a CSV or data table). Fields and columns hold the actual data entries.

Below, we enumerate available record sets and their fields, referencing all entities by their `@id`.

In [ ]:
# List all record sets and their field identifiers
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}  | Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name} | dataType: {field.data_type}")
    print()

# Choose a record set for downstream extraction (here, take the first one as example)
main_record_set_id = record_sets[0].id if record_sets else None

## 3. Data Extraction
Load data from a specific record set (`@id`) into a DataFrame for analysis. Use the record set and field IDs reviewed above.

**Note:** If the dataset contains multiple record sets, you can extract all with a loop. Otherwise, select the main one for demonstration.

In [ ]:
# Extract data from selected record sets
all_record_set_ids = [rs.id for rs in record_sets]
# Dictionary of DataFrames, one per record set
dataframes = dict()

for rec_id in all_record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df

# Show available columns in the main record set
if main_record_set_id:
    print(f"Columns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping/categorization. All field access uses `@id` for column names.

**Example steps:**
- Select a numeric column
- Filter by threshold
- Normalize column
- Group by category

In [ ]:
# ------- EDA Example -------

# For demonstration, select the first numeric field in the record set
df = dataframes.get(main_record_set_id)
numeric_field_id = None
group_field_id = None

if main_record_set_id and df is not None:
    # Find numeric fields in the selected record set by dataType
    rs_obj = next(rs for rs in dataset.record_sets if rs.id == main_record_set_id)
    numeric_types = {'Integer', 'Float', 'Number'}
    for f in rs_obj.fields:
        if f.data_type in numeric_types:
            numeric_field_id = f.id
            break
    # Try to find a non-numeric/groupable field
    for f in rs_obj.fields:
        if f.data_type not in numeric_types:
            group_field_id = f.id
            break

    # Filter: For demonstration, only proceed if numeric_field_id is found
    if numeric_field_id in df.columns:
        if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            # Try to convert to numeric, if possible
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Example: use mean as filter threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in {main_record_set_id} where {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field, if available
        if group_field_id and group_field_id in filtered_df.columns and pd.api.types.is_string_dtype(filtered_df[group_field_id]):
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id]
                .mean()
                .reset_index()
                .sort_values(numeric_field_id, ascending=False)
            )
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in main record set.")

## 5. Visualization
Visualize distributions or relationships in the main record set using column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
if main_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group field found, make a simple boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (RecordSet {main_record_set_id})")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load, explore, and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We referenced all entities (Record Sets, Fields, etc.) by their `@id`, extracted record data into DataFrames, and performed example EDA and visualizations. Further analyses can be built by referencing field and record set `@id`s as needed for your specific use cases.